In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, precision_recall_curve
import joblib

DATA_DIR = Path.home() / "kkbox-churn" / "data" / "parquet"
MODEL_DIR = Path.home() / "kkbox-churn" / "models"

master = pd.read_parquet(DATA_DIR / "master_dataset.parquet")

drop_cols = [
    'msno', 'last_transaction_date', 'last_expire_date',
    'last_listen_date', 'first_listen_date', 'cluster_k5'
]
master = master.drop(columns=drop_cols)

for col in ['city', 'registered_via', 'bd_bucket', 'auto_renew_switch']:
    master[col] = master[col].astype('category')

# Temporal split
master_raw = pd.read_parquet(DATA_DIR / "master_dataset.parquet")
apr_idx = master_raw[master_raw['last_expire_date'].dt.to_period('M') == '2017-04'].index
may_idx = master_raw[master_raw['last_expire_date'].dt.to_period('M') == '2017-05'].index

X = master.drop(columns=['is_churn'])
y = master['is_churn']

X_train, y_train = X.loc[apr_idx], y.loc[apr_idx]
X_val,   y_val   = X.loc[may_idx], y.loc[may_idx]

print(f"Train: {X_train.shape}, churn rate: {y_train.mean():.4f}")
print(f"Val:   {X_val.shape},   churn rate: {y_val.mean():.4f}")
print(f"\nSegments in train:\n{X_train['segment'].value_counts()}")
print(f"\nSegments in val:\n{X_val['segment'].value_counts()}")

Train: (800236, 102), churn rate: 0.0165
Val:   (79942, 102),   churn rate: 0.0502

Segments in train:
segment
At-Risk           389505
Champion          313077
Power_Listener     75536
New_Reengaged      18866
Lost                3252
Name: count, dtype: int64

Segments in val:
segment
Champion          61618
Power_Listener    12127
At-Risk            4277
New_Reengaged      1747
Lost                173
Name: count, dtype: int64


In [2]:
# Cell 2 — Segment-level models
base_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'is_unbalance': True,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

# Combine Lost + At-Risk
X_train_mod = X_train.copy()
X_val_mod = X_val.copy()
X_train_mod['segment'] = X_train_mod['segment'].replace('Lost', 'At-Risk')
X_val_mod['segment']   = X_val_mod['segment'].replace('Lost', 'At-Risk')

segments = ['Champion', 'At-Risk', 'Power_Listener', 'New_Reengaged']
segment_models = {}
results = {}

for seg in segments:
    train_mask = X_train_mod['segment'] == seg
    val_mask   = X_val_mod['segment'] == seg

    X_tr = X_train_mod[train_mask].drop(columns=['segment'])
    y_tr = y_train[train_mask]
    X_vl = X_val_mod[val_mask].drop(columns=['segment'])
    y_vl = y_val[val_mask]

    print(f"\n--- {seg} ---")
    print(f"Train: {len(X_tr):,} users, churn rate: {y_tr.mean():.4f}")
    print(f"Val:   {len(X_vl):,} users, churn rate: {y_vl.mean():.4f}")

    if y_vl.sum() < 10:
        print(f"Skipping — too few churners in val ({y_vl.sum()})")
        continue

    model = lgb.LGBMClassifier(**base_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_vl, y_vl)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=-1)
        ]
    )

    preds = model.predict_proba(X_vl)[:, 1]
    roc = roc_auc_score(y_vl, preds)
    pr  = average_precision_score(y_vl, preds)

    # Optimal threshold
    precisions, recalls, thresholds = precision_recall_curve(y_vl, preds)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    best_thresh = thresholds[f1_scores[:-1].argmax()]
    preds_binary = (preds >= best_thresh).astype(int)

    print(f"ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f} | Threshold: {best_thresh:.4f}")
    print(classification_report(y_vl, preds_binary, digits=3))

    segment_models[seg] = {
        'model': model,
        'threshold': best_thresh
    }
    results[seg] = {'roc': roc, 'pr': pr, 'threshold': best_thresh,
                    'val_size': len(X_vl), 'churn_rate': y_vl.mean()}

    joblib.dump(model, MODEL_DIR / f"lgbm_{seg.lower()}.pkl")

print("\n--- SUMMARY ---")
for seg, res in results.items():
    print(f"{seg:<20} ROC:{res['roc']:.4f}  PR:{res['pr']:.4f}  ChurnRate:{res['churn_rate']:.4f}")


--- Champion ---
Train: 313,077 users, churn rate: 0.0211
Val:   61,618 users, churn rate: 0.0401
ROC-AUC: 0.9599 | PR-AUC: 0.4080 | Threshold: 0.1749
              precision    recall  f1-score   support

           0      0.985     0.952     0.968     59149
           1      0.363     0.658     0.468      2469

    accuracy                          0.940     61618
   macro avg      0.674     0.805     0.718     61618
weighted avg      0.960     0.940     0.948     61618


--- At-Risk ---
Train: 392,757 users, churn rate: 0.0108
Val:   4,450 users, churn rate: 0.2128
ROC-AUC: 0.6447 | PR-AUC: 0.3937 | Threshold: 0.0001
              precision    recall  f1-score   support

           0      0.981     0.241     0.387      3503
           1      0.259     0.983     0.410       947

    accuracy                          0.399      4450
   macro avg      0.620     0.612     0.399      4450
weighted avg      0.828     0.399     0.392      4450


--- Power_Listener ---
Train: 75,536 users,

In [3]:
# Cell 3 — Hybrid prediction: segment models for Champion + Power_Listener, global for rest
import joblib

# Load global model
global_model = joblib.load(MODEL_DIR / "lgbm_churn_final.pkl")

# Get val set without segment column
X_val_no_seg = X_val_mod.drop(columns=['segment'])

# Initialize predictions with global model
hybrid_preds = global_model.predict_proba(X_val_no_seg)[:, 1]
hybrid_preds = pd.Series(hybrid_preds, index=X_val_mod.index)

# Override Champion and Power_Listener with segment models
for seg in ['Champion', 'Power_Listener']:
    val_mask = X_val_mod['segment'] == seg
    X_seg = X_val_mod[val_mask].drop(columns=['segment'])
    seg_preds = segment_models[seg]['model'].predict_proba(X_seg)[:, 1]
    hybrid_preds.loc[val_mask] = seg_preds

# Evaluate hybrid
roc_hybrid = roc_auc_score(y_val, hybrid_preds)
pr_hybrid  = average_precision_score(y_val, hybrid_preds)

# Optimal threshold
precisions, recalls, thresholds = precision_recall_curve(y_val, hybrid_preds)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_thresh_hybrid = thresholds[f1_scores[:-1].argmax()]
preds_hybrid_binary = (hybrid_preds >= best_thresh_hybrid).astype(int)

print(f"{'Metric':<12} {'Global':<10} {'Hybrid'}")
print("-" * 35)
print(f"{'ROC-AUC':<12} 0.9481     {roc_hybrid:.4f}")
print(f"{'PR-AUC':<12} 0.4287     {pr_hybrid:.4f}")
print(f"\nHybrid at optimal threshold ({best_thresh_hybrid:.4f}):")
print(classification_report(y_val, preds_hybrid_binary))

# Save hybrid threshold
joblib.dump(best_thresh_hybrid, MODEL_DIR / "hybrid_threshold.pkl")
print("Saved hybrid threshold.")

Metric       Global     Hybrid
-----------------------------------
ROC-AUC      0.9481     0.9022
PR-AUC       0.4287     0.3731

Hybrid at optimal threshold (0.0234):
              precision    recall  f1-score   support

           0       0.99      0.90      0.94     75930
           1       0.30      0.86      0.45      4012

    accuracy                           0.89     79942
   macro avg       0.65      0.88      0.69     79942
weighted avg       0.96      0.89      0.92     79942

Saved hybrid threshold.
